# Clase 5 — Práctica: de modelado a evaluación

En esta práctica entrenas tu propio árbol de decisión con scikit-learn sobre el dataset de **57.691 recetas de cocina**, tal como lo viste en el notebook de teoría y en el [capítulo 5 del manual](../manual/06_modelado_evaluacion.md).

El propio notebook oficial de IBM sugiere este ejercicio: *"Feel free to select another subset of cuisines and build a decision tree of their recipes. You can select some European cuisines and build decision trees to explore the ingredients that differentiate those cuisines."* Eso es exactamente lo que vas a hacer.

Dos ejercicios:

1. Construir un árbol de decisión con un subconjunto de **cocinas europeas** y evaluarlo con una matriz de confusión.
2. Calcular a mano el accuracy global de tu árbol con `accuracy_score`.

Antes de empezar, ejecuta la celda de abajo: carga y limpia el dataset de recetas exactamente como en clases anteriores, así que ya partes de un `recipes` limpio y listo para usar.

In [ ]:
import pandas as pd
pd.set_option("display.max_columns", None)
import numpy as np
import random
from sklearn import tree
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import itertools

recipes = pd.read_csv(
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBMDeveloperSkillsNetwork-DS0103EN-SkillsNetwork/labs/Module%202/recipes.csv"
)

column_names = recipes.columns.values
column_names[0] = "cuisine"
recipes.columns = column_names
recipes["cuisine"] = recipes["cuisine"].str.lower()

recipes.loc[recipes["cuisine"] == "korea", "cuisine"] = "korean"
recipes.loc[recipes["cuisine"] == "japan", "cuisine"] = "japanese"
recipes.loc[recipes["cuisine"] == "china", "cuisine"] = "chinese"
recipes.loc[recipes["cuisine"] == "thailand", "cuisine"] = "thai"
recipes.loc[recipes["cuisine"] == "india", "cuisine"] = "indian"

recipes_counts = recipes["cuisine"].value_counts()
cuisines_indices = recipes_counts > 50
cuisines_to_keep = list(np.array(recipes_counts.index.values)[np.array(cuisines_indices)])
recipes = recipes.loc[recipes["cuisine"].isin(cuisines_to_keep)]

recipes = recipes.replace(to_replace="Yes", value=1)
recipes = recipes.replace(to_replace="No", value=0)

# Comprueba qué nombres de cocina europea tienes disponibles en el dataset ya limpio
print(sorted(recipes["cuisine"].unique()))
print("Dataset limpio y listo:", recipes.shape)

## Ejercicio 1 — Árbol de decisión con cocinas europeas

Tu tarea es repetir el flujo del notebook de teoría (entrenamiento, train/test, predicción, matriz de confusión), pero esta vez con un subconjunto de **cocinas europeas** en vez de asiáticas e india.

**Pistas:**
- Mira la lista de cocinas que imprimiste arriba y elige entre 4 y 5 cocinas europeas que existan en el dataset (por ejemplo `"french"`, `"italian"`, `"german"`, `"spanish_portuguese"` — usa exactamente los nombres tal como aparecen en la lista impresa, puede que alguno no exista y tengas que ajustar la selección).
- El flujo es: filtrar `recipes` por esas cocinas → separar `ingredients` (columnas 1 en adelante) y `cuisines` (columna "cuisine") → dividir en train/test con `sample_n` recetas por cocina → entrenar un `DecisionTreeClassifier` con el conjunto de entrenamiento → predecir con `.predict()` sobre el conjunto de prueba → dibujar la matriz de confusión.
- Usa `random.seed(1234)` igual que en el notebook de teoría para que tu resultado sea reproducible.
- Si alguna cocina europea tiene pocas recetas, puede que `sample_n = 30` sea demasiado alto: prueba con un valor más bajo si te da error de muestra insuficiente.

In [ ]:
# TODO: elige entre 4 y 5 cocinas europeas presentes en `recipes` (usa la lista impresa arriba)
european_cuisines = None  # TODO: lista de strings, por ejemplo ["french", "italian", "german", "spanish_portuguese"]

# TODO: filtra `recipes` para quedarte solo con esas cocinas
european_recipes = None

# TODO: separa la variable a predecir (columna "cuisine") de las variables de entrada (el resto de columnas)
cuisines = None
ingredients = None

# TODO: define cuántas recetas de cada cocina van al conjunto de prueba
sample_n = None  # por ejemplo 15 o 20, según cuántas recetas tenga la cocina con menos ejemplos

random.seed(1234)

# TODO: construye el conjunto de prueba muestreando `sample_n` recetas de cada cocina
test_set = None

test_ingredients = None
test_cuisines = None

# TODO: construye el conjunto de entrenamiento con todo lo que NO quedó en el conjunto de prueba
train_set = None
train_ingredients = None
train_cuisines = None

# TODO: crea y entrena un DecisionTreeClassifier con el conjunto de entrenamiento
euro_tree = None

# TODO: predice las cocinas del conjunto de prueba con .predict()
euro_pred_cuisines = None

print("Cocinas usadas:", european_cuisines)

In [ ]:
# TODO: usa confusion_matrix para comparar test_cuisines contra euro_pred_cuisines,
# conviértela a porcentaje por fila, y dibújala con plt.imshow (igual que en el notebook de teoría)

labels = None  # TODO: np.unique(test_cuisines)
conf_matrix = None  # TODO: confusion_matrix(test_cuisines, euro_pred_cuisines, labels=labels)

# TODO: convierte conf_matrix a porcentaje por fila y dibújala con plt.imshow,
# ponle título, colorbar, ticks con los nombres de cocina, y los valores como texto en cada celda

<details>
<summary>💡 Ver solución</summary>

```python
# Elegimos 4 cocinas europeas presentes en el dataset limpio
european_cuisines = ["french", "italian", "german", "spanish_portuguese"]

european_recipes = recipes[recipes.cuisine.isin(european_cuisines)]

cuisines = european_recipes["cuisine"]
ingredients = european_recipes.iloc[:, 1:]

# Usamos un sample_n moderado por si alguna cocina europea tiene pocas recetas
sample_n = 20

random.seed(1234)
test_set = european_recipes.groupby("cuisine", group_keys=False).apply(lambda x: x.sample(sample_n))

test_ingredients = test_set.iloc[:, 1:]
test_cuisines = test_set["cuisine"]

test_index = european_recipes.index.isin(test_set.index)
train_set = european_recipes[~test_index]
train_ingredients = train_set.iloc[:, 1:]
train_cuisines = train_set["cuisine"]

euro_tree = tree.DecisionTreeClassifier(max_depth=15)
euro_tree.fit(train_ingredients, train_cuisines)

euro_pred_cuisines = euro_tree.predict(test_ingredients)

print("Cocinas usadas:", european_cuisines)

# Matriz de confusión
labels = np.unique(test_cuisines)
conf_matrix = confusion_matrix(test_cuisines, euro_pred_cuisines, labels=labels)
conf_matrix = (conf_matrix.astype("float") / conf_matrix.sum(axis=1)[:, np.newaxis]) * 100

plt.figure(figsize=(8, 6))
plt.imshow(conf_matrix, interpolation="nearest", cmap=plt.cm.Blues)
plt.title("Matriz de confusión — cocinas europeas")
plt.colorbar()
tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)

thresh = conf_matrix.max() / 2.0
for i, j in itertools.product(range(conf_matrix.shape[0]), range(conf_matrix.shape[1])):
    plt.text(j, i, format(conf_matrix[i, j], ".2f"),
             horizontalalignment="center",
             color="white" if conf_matrix[i, j] > thresh else "black")

plt.tight_layout()
plt.ylabel("True label")
plt.xlabel("Predicted label")
plt.show()
```

Nota: como el árbol se entrena con muestreo aleatorio, tus cocinas confundidas entre sí pueden variar respecto a otros compañeros, pero el patrón general (algunas cocinas europeas comparten muchos ingredientes, como francesa e italiana) debería sostenerse.

</details>

## Ejercicio 2 — Calcula el accuracy global a mano

Ahora que tienes `test_cuisines` (las cocinas reales) y `euro_pred_cuisines` (las cocinas predichas) del ejercicio 1, calcula el accuracy global de tu árbol.

**Pistas:**
- `accuracy_score(y_real, y_predicho)` de `sklearn.metrics` devuelve la proporción de predicciones correctas sobre el total.
- Compara ese número contra el accuracy del árbol de recetas asiáticas/india del notebook de teoría (aproximadamente 60-77% de aciertos por cocina). ¿Tu árbol de cocinas europeas acierta más, menos o parecido?

In [ ]:
# TODO: calcula el accuracy global comparando test_cuisines contra euro_pred_cuisines
accuracy_global = None

print(f"Accuracy global del árbol de cocinas europeas: {accuracy_global:.2%}")

<details>
<summary>💡 Ver solución</summary>

```python
# accuracy_score compara las predicciones contra los valores reales y devuelve
# la proporción de aciertos (entre 0 y 1)
accuracy_global = accuracy_score(test_cuisines, euro_pred_cuisines)

print(f"Accuracy global del árbol de cocinas europeas: {accuracy_global:.2%}")
```

Compara este número con la accuracy global calculada en el notebook de teoría para el árbol de cocinas asiáticas e india. No hay un número "correcto" único: lo importante es que entiendas que este es el mismo tipo de métrica que viste en la tabla del caso CHF (columna "Accuracy global"), y que un solo número global no te dice nada sobre qué cocinas concretas se confunden entre sí — para eso necesitas la matriz de confusión del ejercicio 1.

</details>

## Para cerrar

Si completaste los dos ejercicios, ya puedes:

- Entrenar un `DecisionTreeClassifier` de scikit-learn sobre cualquier subconjunto de clases de un dataset tabular.
- Dividir datos en entrenamiento y prueba, predecir con `.predict()`, y leer una matriz de confusión real.
- Calcular el accuracy global de un modelo con `accuracy_score` y entender por qué ese único número nunca cuenta toda la historia.

En la próxima clase vas a ver qué pasa después de tener un modelo evaluado y aceptado: cómo se despliega en el mundo real y cómo el feedback lo mejora con el tiempo.